In [4]:
import pandas as pd

nav = pd.read_feather("Mutual_Fund_Nav.feather")

# 打印每列类型
for i, c in enumerate(nav.columns):
    print(i, c, type(c))

0 F_INFO_WINDCODE <class 'str'>
1 ANN_DATE <class 'str'>
2 PRICE_DATE <class 'str'>
3 F_NAV_UNIT <class 'str'>
4 F_NAV_DIVACCUMULATED <class 'str'>
5 F_NAV_ADJFACTOR <class 'str'>
6 F_PRT_NETASSET <class 'str'>
7 F_ASSET_MERGEDSHARESORNOT <class 'str'>
8 NETASSET_TOTAL <class 'str'>
9 F_NAV_ADJUSTED <class 'str'>
10 IS_EXDIVIDENDDATE <class 'str'>
11 F_NAV_DISTRIBUTION <class 'str'>
12 S_INFO_ASHARECODE <class 'str'>
13 CUM_NET_ASSET_VALUE <class 'str'>


给基金分类

设置函数

In [1]:
import pandas as pd
import numpy as np
import re


# 优先级（越靠前优先级越高）
FUND_TYPE_PRIORITY = [
    "FOF",
    "ETF",
    "LOF",
    "QDII",
    "Money Market Fund",
    "Bond Fund",
    "Equity Fund",      # 包含股票型 + 偏股混合 + 选股基金
    "Hybrid Fund",
    "Other/Unknown"
]


def classify_one_type(text: str,fund_code: str = "") -> str:

    if pd.isna(text):
        return "Other/Unknown"

    x = str(text).upper()
    fund_code = str(fund_code)

     
    # 1. FOF
    # ETF-FOF / Pension FOF / MOM 全部归 FOF
    if re.search(
        r"FOF|FUND OF FUNDS|PENSION|TARGET DATE|TARGET RISK|MOM|MANAGER OF MANAGERS",
        x
    ):
        return "FOF"
    
    # 2. Passive Index Fund
    if re.search(
        r"INDEX FUND|INDEX|PASSIVE|TRACKING|TRACKER|"
        r"CSI ?300|CSI ?500|SSE ?50|NASDAQ|S&P ?500|"
        r"MSCI|HANG SENG|HS300|ETF LINKED",
        x
    ):
        return "Passive Index Fund"

     
    # 2. ETF
    if ( 
        fund_code.startswith(tuple(["510","511","512","513","515","516","517","518","520","530","551","560","561","562","563","588","589","159"]))
        or
        re.search(
            r"\bETF\b|EXCHANGE TRADED FUND",
            x
        )
    ):
        return "ETF"

    # 3. LOF
    if re.search(
        r"\bLOF\b|LISTED OPEN[- ]?ENDED",
        x
    ):
        return "LOF"

    # 4. QDII
    if re.search(
        r"\bQDII\b|QUALIFIED DOMESTIC INSTITUTIONAL INVESTOR",
        x
    ):
        return "QDII"

    # 5. Money Market Fund
    if re.search(
        r"MONEY MARKET|MONEY FUND|CASH FUND",
        x
    ):
        return "Money Market Fund"

    # 6. Bond Fund
    if re.search(
        r"BOND FUND|BOND|FIXED INCOME|CONVERTIBLE BOND|SHORT[- ]?TERM BOND",
        x
    ):
        return "Bond Fund"

    # 7. Equity Fund
    #
    # 把：
    # - 股票型
    # - 偏股混合
    # - 选股基金
    # 全部合并
    if re.search(
        r"EQUITY|STOCK FUND|STOCK|EQUITY FUND|ACTIVE EQUITY|"
        r"AGGRESSIVE ALLOCATION|PARTIAL EQUITY|"
        r"FLEXIBLE ALLOCATION|SELECTION FUND|"
        r"SHARE FUND|GROWTH FUND",
        x
    ):
        return "Equity Fund"

    # 8. Hybrid Fund
    # 剩余混合型

    if re.search(
        r"HYBRID|BALANCED|MIXED|ALLOCATION FUND",
        x
    ):
        return "Hybrid Fund"

    return "Other/Unknown"


def classify_fund_main_type(industry_df,
                             fund_sector_df,
                             current_only=True):

    industry = industry_df.copy()
    fund_sector = fund_sector_df.copy()

    # 转字符串
    industry["INDUSTRIESCODE"] = industry["INDUSTRIESCODE"].astype(str)
    fund_sector["S_INFO_SECTOR"] = fund_sector["S_INFO_SECTOR"].astype(str)

    # 只保留有效分类
    if "USED" in industry.columns:
        industry = industry[industry["USED"] == 1]

    # 只保留当前分类
    if current_only and "CUR_SIGN" in fund_sector.columns:
        fund_sector = fund_sector[fund_sector["CUR_SIGN"] == 1]

    # 合并
    df = fund_sector.merge(
        industry,
        left_on="S_INFO_SECTOR",
        right_on="INDUSTRIESCODE",
        how="left"
    )

    # 可用于分类的文本字段
    text_cols = [
        "INDUSTRIESNAME",
        "INDUSTRIESALIAS",
        "MEMO",
        "CHINESEDEFINITION",
        "WIND_NAME_ENG"
    ]

    # 防止缺失字段
    for col in text_cols:
        if col not in df.columns:
            df[col] = ""

    # 拼接文本
    df["type_text"] = (
        df[text_cols]
        .fillna("")
        .astype(str)
        .agg(" ".join, axis=1)
    )

    # 分类
    df["fund_type"] = df.apply(
    lambda row: classify_one_type(
        row["type_text"],
        row["F_INFO_WINDCODE"]
    ),
    axis=1
)

    # 优先级映射
    priority_map = {
        v: i for i, v in enumerate(FUND_TYPE_PRIORITY)
    }

    # 一个基金可能对应多个分类
    # 取优先级最高的那个
    def choose_main_type(types):

        types = list(set(types))

        if len(types) == 0:
            return "Other/Unknown"

        return sorted(
            types,
            key=lambda x: priority_map.get(x, 999)
        )[0]

    result = (
        df.groupby("F_INFO_WINDCODE")
        .agg(
            main_type=("fund_type", choose_main_type),

            all_types=(
                "fund_type",
                lambda x: sorted(set(x))
            ),

            matched_sector_names=(
                "INDUSTRIESNAME",
                lambda x: sorted(set(
                    x.dropna().astype(str)
                ))
            ),

            sector_codes=(
                "S_INFO_SECTOR",
                lambda x: sorted(set(
                    x.dropna().astype(str)
                ))
            )
        )
        .reset_index()
    )

    return result, df

运行函数

In [3]:
import pandas as pd
df1 = pd.read_csv('ASHAREINDUSTRIESCODE_202605221326.csv') #行业分类定义表（行业字典表）
df2 = pd.read_csv('CHINAMUTUALFUNDSECTOR_202605221321.csv') #基金-行业映射表（成分归属表）
fund_type_result, detail = classify_fund_main_type(
    industry_df=df1,
    fund_sector_df=df2,
    current_only=False
)

分类存入类别文件中

In [4]:
import pandas as pd


# 1. 读取数据

# NAV数据

nav_df = pd.read_feather("Mutual_Fund_Nav.feather")


In [5]:

# 2. 统一基金代码格式

fund_type_result["F_INFO_WINDCODE"] = (
    fund_type_result["F_INFO_WINDCODE"]
    .astype(str)
    .str.upper()
)

nav_df["F_INFO_WINDCODE"] = (
    nav_df["F_INFO_WINDCODE"]
    .astype(str)
    .str.upper()
)


In [6]:

# 3. 合并基金类型

nav_with_type = nav_df.merge(
    fund_type_result[
        ["F_INFO_WINDCODE", "main_type"]
    ],
    left_on="F_INFO_WINDCODE",
    right_on="F_INFO_WINDCODE",
    how="left"
)


In [9]:
nav_with_type['ANN_DATE']

0           19981130
1           19981207
2           19981214
3           19981221
4           19981228
              ...   
35375157    20250621
35375158    20250422
35375159    20240111
35375160    20260303
35375161    20240518
Name: ANN_DATE, Length: 35375162, dtype: int64

In [10]:

# 4. 按基金类型筛选

# ETF
etf_nav = nav_with_type[
    nav_with_type["main_type"] == "ETF"
].copy()

# FOF
fof_nav = nav_with_type[
    nav_with_type["main_type"] == "FOF"
].copy()

# 股票型 + 偏股混合型
equity_nav = nav_with_type[
    nav_with_type["main_type"] == "Equity Fund"
].copy()

# 其他混合型
mixed_nav = nav_with_type[
    nav_with_type["main_type"] == "Hybrid Fund"
].copy()

# 债券型
bond_nav = nav_with_type[
    nav_with_type["main_type"] == "Bond Fund"
].copy()

# QDII
qdii_nav = nav_with_type[
    nav_with_type["main_type"] == "QDII"
].copy()

# 货币基金
money_nav = nav_with_type[
    nav_with_type["main_type"] == "Money Market Fund"
].copy()

# 未识别
unknown_nav = nav_with_type[
    nav_with_type["main_type"].isna()
].copy()

# 5. 保存 feather 文件

etf_nav.reset_index(drop=True).to_feather(
    "ETF_nav.feather"
)

fof_nav.reset_index(drop=True).to_feather(
    "FOF_nav.feather"
)

equity_nav.reset_index(drop=True).to_feather(
    "股票型_偏股混合型_nav.feather"
)

mixed_nav.reset_index(drop=True).to_feather(
    "其他混合型_nav.feather"
)

bond_nav.reset_index(drop=True).to_feather(
    "债券型_nav.feather"
)

qdii_nav.reset_index(drop=True).to_feather(
    "QDII_nav.feather"
)

money_nav.reset_index(drop=True).to_feather(
    "货币基金_nav.feather"
)

unknown_nav.reset_index(drop=True).to_feather(
    "其他未知_nav.feather"
)

In [11]:
equity_nav['ANN_DATE']

0           19981130
1           19981207
2           19981214
3           19981221
4           19981228
              ...   
35375151    20180904
35375152    20180904
35375153    20180904
35375154    20110427
35375158    20250422
Name: ANN_DATE, Length: 13577142, dtype: int64

标记出基金规模大于1亿的

In [ ]:
import pandas as pd
df1 = pd.read_feather("交易日偏股型基金.feather")

df1 = df1.sort_values(['F_INFO_WINDCODE', 'ANN_DATE'])

# 先向前填充规模
df1['F_PRT_NETASSET'] = (
    df1.groupby('F_INFO_WINDCODE')['F_PRT_NETASSET']
       .ffill()
)

# 每个日期对应的规模是否大于1亿
df1['1M'] = df1['F_PRT_NETASSET'] > 1e8


In [6]:
df1.reset_index(drop=True).to_feather('交易日偏股型基金.feather')

In [15]:
# 查看数量统计

summary = {
    "ETF": etf_nav['F_INFO_WINDCODE'].nunique(),
    "FOF": fof_nav['F_INFO_WINDCODE'].nunique(),
    "股票型+偏股混合型": equity_nav['F_INFO_WINDCODE'].nunique(),
    "其他混合型": mixed_nav['F_INFO_WINDCODE'].nunique(),
    "债券型": bond_nav['F_INFO_WINDCODE'].nunique(),
    "QDII": qdii_nav['F_INFO_WINDCODE'].nunique(),
    "货币基金": money_nav['F_INFO_WINDCODE'].nunique(),
    "未知": unknown_nav['F_INFO_WINDCODE'].nunique(),
}

print(pd.Series(summary))

ETF           1678
FOF           1325
股票型+偏股混合型    10300
其他混合型         1963
债券型           8367
QDII           416
货币基金          1158
未知               6
dtype: int64


Similarly，我们把这个分类方式也应用到持仓数据上

In [7]:
import pandas as pd
# 读取 StockHold 数据
hold1 = pd.read_csv("CMFOTHERPORTFOLIO_202605200918.csv")
hold2 = pd.read_csv("CHINAMUTUALFUNDSTOCKPORTFOLIO_202605190939.csv")
hold3 = pd.read_csv("CHINAMUTUALFUNDBONDPORTFOLIO_202605200915.csv")

In [8]:

# 1. 构建 基金代码 -> 基金类型 映射


fund_type_map = dict(
    zip(
        fund_type_result["F_INFO_WINDCODE"],
        fund_type_result["main_type"]
    )
)



# 2. 给三个持仓表打标签


for df in [hold1, hold2, hold3]:

    # 添加基金类型
    df["fund_type"] = df["S_INFO_WINDCODE"].map(
        fund_type_map
    )

    # 是否属于 Equity Fund
    df["is_equity_fund"] = (
        df["fund_type"] == "Equity Fund"
    )



# 3. 查看结果

print(hold1.groupby("fund_type")[
    "S_INFO_WINDCODE"
].nunique())
print(hold2.groupby("fund_type")[
    "S_INFO_WINDCODE"
].nunique())
print(hold3.groupby("fund_type")[
    "S_INFO_WINDCODE"
].nunique())

fund_type
Bond Fund        1100
ETF               215
Equity Fund      1246
FOF              1211
Hybrid Fund       450
Other/Unknown    2537
Name: S_INFO_WINDCODE, dtype: int64
fund_type
Bond Fund             2193
ETF                   1468
Equity Fund          10052
FOF                    642
Hybrid Fund           1908
Money Market Fund        1
Other/Unknown         3457
QDII                     2
Name: S_INFO_WINDCODE, dtype: int64
fund_type
Bond Fund            8198
ETF                   683
Equity Fund          8357
FOF                  1101
Hybrid Fund          1934
Money Market Fund    1143
Other/Unknown        3525
QDII                    9
Name: S_INFO_WINDCODE, dtype: int64


In [9]:
hold1.to_feather('CMFOTHERPORTFOLIO.feather')
hold2.to_feather('CHINAMUTUALFUNDSTOCKPORTFOLIO.feather')
hold3.to_feather('CHINAMUTUALFUNDBONDPORTFOLIO.feather')

In [10]:
# 导入开放式基金NAV数据
df = pd.read_feather('股票型_偏股混合型_nav.feather')
# 保证ANNDATE格式为datetime
df['ANN_DATE'] = pd.to_datetime(df['ANN_DATE'], format='%Y%m%d', errors='coerce')
# 剔除出20260201以后才成立的基金
earliest_dates = df.groupby('F_INFO_WINDCODE')['ANN_DATE'].min().reset_index()
cutoff_date = pd.to_datetime('2026-02-01')
fund_to_remove = earliest_dates[earliest_dates['ANN_DATE']>=cutoff_date]['F_INFO_WINDCODE']
remove_set = set(fund_to_remove)
# nav数据中的基金编号集
nav_funds = set(df['F_INFO_WINDCODE'])

SH_funds = set(pd.concat((hold1['S_INFO_WINDCODE'],hold2['S_INFO_WINDCODE'],hold3['S_INFO_WINDCODE'])))
# NAV中有，但持仓文件没有
missing_funds = nav_funds - SH_funds
# missing基金中有，而需要剔除的数据中没有的
new_missing_funds = missing_funds-remove_set 

print(f"未剔除20260201日期后才建立的基金前缺失的基金数量: {len(missing_funds)}")
print(missing_funds)
print(f"剔除新成立基金后确定的缺失基金数量: {len(new_missing_funds)}")
print(new_missing_funds)

未剔除20260201日期后才建立的基金前缺失的基金数量: 252
{'026947.OF', '026732.OF', '004491.OF', '026395.OF', '010650.OF', '026904.OF', '026619.OF', '026511.OF', '010672.OF', '026891.OF', '027011.OF', '026557.OF', '026808.OF', '026837.OF', '026819.OF', '026994.OF', '026006.OF', '017543.OF', '026778.OF', '026842.OF', '026728.OF', '027000.OF', '026686.OF', '010682.OF', '025090.OF', '027189.OF', '026476.OF', '026854.OF', '027049.OF', '026540.OF', '026866.OF', '027062.OF', '027068.OF', '027364.OF', '026466.OF', '026268.OF', '026223.OF', '026933.OF', '026374.OF', '026858.OF', '026872.OF', '026941.OF', '026710.OF', '027098.OF', '026262.OF', '026465.OF', '026679.OF', '026643.OF', '026521.OF', '026533.OF', '026763.OF', '026937.OF', '026803.OF', '026733.OF', '004508.OF', '010686.OF', '026839.OF', '026887.OF', '026890.OF', '026990.OF', '026762.OF', '027271.OF', '025786.OF', '026462.OF', '016000.OF', '026464.OF', '026541.OF', '026731.OF', '027156.OF', '026376.OF', '004492.OF', '026635.OF', '026430.OF', '025877.OF', '02

In [ ]:
import pandas as pd
from pathlib import Path

base_dir = Path('.')
feather_path = base_dir / '交易日偏股型基金.feather'
csv_path = base_dir / 'CHINAMUTUALFUNDDESCRIPTION_202606031717.csv'

if not feather_path.exists() or not csv_path.exists():
    base_dir = Path('基金数据')
    feather_path = base_dir / '交易日偏股型基金.feather'
    csv_path = base_dir / 'CHINAMUTUALFUNDDESCRIPTION_202606031717.csv'

type_col = 'F_INFO_FIRSTINVESTTYPE'
company_col = 'F_INFO_CORP_FUNDMANAGEMENTCOMP'
keep_types = {'股票型', '混合型'}

df_fund = pd.read_feather(feather_path)
desc_cols = ['F_INFO_WINDCODE', type_col, company_col]
df_desc = pd.read_csv(csv_path, usecols=desc_cols, dtype=str)

df_fund['F_INFO_WINDCODE'] = df_fund['F_INFO_WINDCODE'].astype(str).str.strip().str.upper()
df_desc['F_INFO_WINDCODE'] = df_desc['F_INFO_WINDCODE'].astype(str).str.strip().str.upper()
df_desc[type_col] = df_desc[type_col].astype('string').str.strip()
df_desc[company_col] = df_desc[company_col].astype('string').str.strip()

# CSV 是基金维表；先按基金代码去重，再合并回长表。
fund_ref = (
    df_desc.dropna(subset=['F_INFO_WINDCODE'])
    .drop_duplicates(subset='F_INFO_WINDCODE', keep='last')
    .rename(columns={type_col: 'csv_main_type', company_col: 'fund_com'})
)

before_funds = df_fund['F_INFO_WINDCODE'].nunique()
before_rows = len(df_fund)

df_checked = df_fund.merge(
    fund_ref[['F_INFO_WINDCODE', 'csv_main_type', 'fund_com']],
    on='F_INFO_WINDCODE',
    how='left',
    validate='many_to_one',
)

matched_mask = df_checked['csv_main_type'].notna()
matched_funds = df_checked.loc[matched_mask, 'F_INFO_WINDCODE'].nunique()
missing_funds = before_funds - matched_funds

keep_mask = df_checked['csv_main_type'].isin(keep_types)
removed_non_target_funds = df_checked.loc[matched_mask & ~keep_mask, 'F_INFO_WINDCODE'].nunique()
removed_missing_ref_funds = df_checked.loc[~matched_mask, 'F_INFO_WINDCODE'].nunique()

df_checked = df_checked.loc[keep_mask].copy()
df_checked['main_type'] = df_checked['csv_main_type']
df_checked = df_checked.drop(columns=['csv_main_type'])

after_funds = df_checked['F_INFO_WINDCODE'].nunique()
after_rows = len(df_checked)

df_checked.reset_index(drop=True).to_feather(feather_path)

print(f'原 feather 长表记录数: {before_rows:,}')
print(f'原 feather 基金数: {before_funds:,}')
print(f'在 CSV 中匹配到类型/基金公司的基金数: {matched_funds:,}')
print(f'CSV 中未匹配到的基金数，已剔除: {missing_funds:,}')
print(f'基金类型不是股票型/混合型的基金数，已剔除: {removed_non_target_funds:,}')
print(f'因缺少 CSV 参考被剔除的基金数: {removed_missing_ref_funds:,}')
print(f'最终保留基金数: {after_funds:,}')
print(f'最终保留长表记录数: {after_rows:,}')
print('main_type 已按 CSV 的 F_INFO_FIRSTINVESTTYPE 更新，fund_com 已按 CSV 的 F_INFO_CORP_FUNDMANAGEMENTCOMP 增加。')

In [9]:
import pandas as pd
from pathlib import Path

base_dir = Path('.')
feather_path = base_dir / '交易日偏股型基金.feather'
csv_path = base_dir / 'CHINAMUTUALFUNDDESCRIPTION_202606031717.csv'

if not feather_path.exists() or not csv_path.exists():
    base_dir = Path('基金数据')
    feather_path = base_dir / '交易日偏股型基金.feather'
    csv_path = base_dir / 'CHINAMUTUALFUNDDESCRIPTION_202606031717.csv'

df_fund = pd.read_feather(feather_path)
df_type = pd.read_csv(
    csv_path,
    usecols=['F_INFO_WINDCODE', 'F_INFO_TYPE'],
    dtype=str,
)

df_fund['F_INFO_WINDCODE'] = df_fund['F_INFO_WINDCODE'].astype(str).str.strip().str.upper()
df_type['F_INFO_WINDCODE'] = df_type['F_INFO_WINDCODE'].astype(str).str.strip().str.upper()
df_type['F_INFO_TYPE'] = df_type['F_INFO_TYPE'].astype('string').str.strip()

fund_type_ref = (
    df_type.dropna(subset=['F_INFO_WINDCODE'])
    .drop_duplicates(subset='F_INFO_WINDCODE', keep='last')
)

before_funds = df_fund['F_INFO_WINDCODE'].nunique()
before_rows = len(df_fund)

if 'F_INFO_TYPE' in df_fund.columns:
    df_fund = df_fund.drop(columns=['F_INFO_TYPE'])

df_fund = df_fund.merge(
    fund_type_ref,
    on='F_INFO_WINDCODE',
    how='left',
    validate='many_to_one',
)

matched_funds = df_fund.loc[df_fund['F_INFO_TYPE'].notna(), 'F_INFO_WINDCODE'].nunique()
missing_funds = before_funds - matched_funds

df_fund.reset_index(drop=True).to_feather(feather_path)

print(f'处理 feather 长表记录数: {before_rows:,}')
print(f'处理基金数: {before_funds:,}')
print(f'成功匹配 F_INFO_TYPE 的基金数: {matched_funds:,}')
print(f'未匹配到 F_INFO_TYPE 的基金数: {missing_funds:,}')
print('已按 F_INFO_WINDCODE 增加/刷新 F_INFO_TYPE 列，并保存回交易日偏股型基金.feather。')

处理 feather 长表记录数: 13,266,734
处理基金数: 10,183
成功匹配 F_INFO_TYPE 的基金数: 10,183
未匹配到 F_INFO_TYPE 的基金数: 0
已按 F_INFO_WINDCODE 增加/刷新 F_INFO_TYPE 列，并保存回交易日偏股型基金.feather。
